# 0. Onboarding — your first agent, running on local Qwen

A `smolagents` **CodeAgent**: the model (`qwen2:7b`, served by Ollama) does not fill in a JSON form,
it **writes Python code** that gets executed. Then we look at what stops that code from doing damage.

Before running: Ollama must be up (`ollama list` shows `qwen2:7b`), and the kernel must be this project's `.venv`.

## 1. The brain: your local model, served by Ollama

In [ ]:
from smolagents import CodeAgent, LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",    # must match a name from `ollama list`
    api_base="http://127.0.0.1:11434",  # where Ollama listens on your machine
    num_ctx=8192,                       # context window: how much text the model "sees" at once
    temperature=0,                      # randomness: 0 = deterministic, 1 = random
    # 0 = always the most likely token: repeatable, follows the format better.
)

## 2. The agent: wraps the brain and lets it write + run Python

In [ ]:
agent = CodeAgent(
    tools=[],       # no extra tools yet: the agent can still write code
    model=model,
    max_steps=6,    # safety net: stop after 6 steps instead of the default 20 (catches loops early)
)

## 3. Give it tasks that benefit from running code

Sections 3 and 4: each cell calls the model: it takes a while, and the trace shows the code the agent writes at every step.
Run only the ones you need (in the `.py` they were switched off by wrapping them in a `"""` string: in a notebook, you just don't run the cell).

In [ ]:
# --- 3. A first task ---
# result = agent.run("what are the key ingredients of pizza margherita?")
result = agent.run("What is the 10th Fibonacci number?")
print(result)

## 4. A second task: counting + summing, easier in code than in words

In [ ]:
# --- 4. A second task ---
result2 = agent.run("How many prime numbers are there below 100, and what is their sum?")
print(result2)

## 5. Custom executor: restrict what the agent's code can do (smolagents)

The `CodeAgent` does **not** pass the code to Python's `exec`: its `LocalPythonExecutor` walks the code
and runs only what is allowed. Here we use it directly, with no model, and feed it harmful commands.

Where this sits on the safety spectrum (see `brain/1.0-CS-agents.md`):
**default `LocalPythonExecutor` with fine-grained authorizations** — a bouncer at the door, not a wall.
Real isolation = Docker / E2B.

`numpy` must be installed in the `.venv` (`pip install numpy`, in bash):
the executor checks right away that every authorized import really exists.

In [ ]:
from smolagents.local_python_executor import LocalPythonExecutor

# --- 5a. Custom executor: the default safe imports + "numpy" ---
custom_executor = LocalPythonExecutor(["numpy"])


# --- 5b. Run a command, print the error instead of crashing ---
def run_capture_exception(command: str):
    try:
        custom_executor(command)
    except Exception as e:
        print("ERROR:\n", e)

In [ ]:
# --- 5c. Not Python at all (a shell command): it just does not parse ---
harmful_command = "!echo Bad command"
run_capture_exception(harmful_command)
# >>> ERROR: invalid syntax (<unknown>, line 1)

In [ ]:
# --- 5d. Imports like os are blocked unless explicitly added to the authorized imports ---
harmful_command = "import os; exit_code = os.system('echo Bad command')"
run_capture_exception(harmful_command)
# >>> ERROR: ... Import of os is not allowed. Authorized imports are: [..., 'numpy', ...]

In [ ]:
# --- 5e. Even through an authorized module, a forbidden one stays forbidden ---
harmful_command = "import random; random._os.system('echo Bad command')"
run_capture_exception(harmful_command)
# >>> ERROR: ... Forbidden access to module: os

In [ ]:
# --- 5f. Infinite loops are interrupted after N operations ---
harmful_command = """
while True:
    pass
"""
run_capture_exception(harmful_command)
# >>> ERROR: ... Maximum number of 1000000 iterations in While loop exceeded

## 6. Sandboxes: run the agent's code somewhere else

Section 5 is a **rulebook** on your machine. A **sandbox** is a **fence**: the code runs in an isolated place,
and whatever it breaks, it breaks there. One argument switches it: `executor_type=...`.

📚 Source: smolagents documentation, [“Secure code execution”](https://huggingface.co/docs/smolagents/tutorials/secure_code_execution).
Full summary in `brain/1.5-CS-sandbox-approaches.md`.

### 6.1 Different types of sandboxes

**Two approaches.** With `executor_type=...` only the **code snippets** go into the sandbox (approach 1):
the agent, the model calls and your API keys stay on your machine, and only the output comes back.
Running the **entire agentic system** inside the sandbox (approach 2) isolates everything and supports multi-agents,
but needs manual setup and your keys must travel into the sandbox.

```
APPROACH 1 (executor_type)              APPROACH 2 (manual)
your machine ──code──▶ sandbox          your machine ──starts──▶ sandbox: agent + model calls + tools + code
             ◀─output─                                 ──keys───▶
```

**Four sandboxes**, all with approach 1:

| `executor_type` | Where the code runs | You need |
| --- | --- | --- |
| `"local"` (default) | your machine, in `LocalPythonExecutor` (section 5) | nothing: a rulebook, not a fence |
| `"blaxel"` | cloud VMs, blaxel.ai: wake from hibernation in under 25 ms | account + `pip install 'smolagents[blaxel]'` + `BL_API_KEY`, `BL_WORKSPACE` (or `bl login`) |
| `"e2b"` | cloud sandboxes, e2b.dev | account + `pip install 'smolagents[e2b]'` + `E2B_API_KEY` |
| `"modal"` | cloud sandboxes, modal.com | account + `pip install 'smolagents[modal]'` + a Modal token |
| `"docker"` | a container **on your machine**: no cloud, no account | Docker installed + `pip install 'smolagents[docker]'` |

**Cloud vs Docker.** Cloud: only a `pip` extra, but an account, a key, the code leaves your machine, and it may cost money.
Docker: all local, but Docker itself must be installed (on WSL a small project): **deferred** until the agent gets tools that touch files or the web.

**Always `with`:** the context manager shuts the sandbox down when the block ends, even after an error
(or call `agent.cleanup()`). A sandbox left running keeps eating resources, and in the cloud, money.

**Which model?** The docs use `InferenceClientModel()` (Hugging Face API, needs `HF_TOKEN` in `.env`).
In approach 1 the model is called **from your machine**, so in principle the local `model` of section 1 (qwen via Ollama) works too:
we pass it as a comment to try. *Not tested yet: no accounts.*

⚠️ Until the account and key exist, these cells **fail with an authentication error**: that is expected (see section 7 of the cheat sheet).

In [ ]:
# --- 6.2 Blaxel ---
# Needs: blaxel.ai account + BL_API_KEY and BL_WORKSPACE (or `bl login`)
from smolagents import InferenceClientModel, CodeAgent

# Using the agent as a context manager (the `with` statement) ensures that the Blaxel sandbox
# is cleaned up immediately after the agent completes its task.
# Alternatively, you can manually call the agent's cleanup() method.
with CodeAgent(model=InferenceClientModel(), tools=[], executor_type="blaxel") as agent:   # or model=model (local qwen)
    agent.run("Can you give me the 100th Fibonacci number?")

In [ ]:
# --- 6.3 E2B ---
# Needs: e2b.dev account + E2B_API_KEY
from smolagents import InferenceClientModel, CodeAgent

with CodeAgent(model=InferenceClientModel(), tools=[], executor_type="e2b") as agent:   # or model=model (local qwen)
    agent.run("Can you give me the 100th Fibonacci number?")

In [ ]:
# --- 6.4 Modal ---
# Needs: modal.com account + a Modal token
from smolagents import InferenceClientModel, CodeAgent

with CodeAgent(model=InferenceClientModel(), tools=[], executor_type="modal") as agent:   # or model=model (local qwen)
    agent.run("What is the 42th Fibonacci number?")